# Cinéfilo: base de conhecimento em RDF, RDFS e OWL

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolas-Kleiton/sbc-cinefilo-ontologia/blob/main/Cinefilo_Ontologia.ipynb)

**Mini-Projeto 3, Sistemas Baseados em Conhecimento (UFPB).**

Nos Mini-Projetos 1 e 2, o Cinéfilo recomendava filmes com regras nítidas e com lógica fuzzy. Aqui o
mesmo domínio vira um **grafo de conhecimento**: filmes, profissionais, estúdios e prêmios descritos
em Turtle no arquivo [`cinema.ttl`](cinema.ttl), com hierarquia de classes em RDFS e axiomas em OWL.

O notebook carrega a base com `rdflib` e executa todas as consultas:

| Seção | Conteúdo |
|---|---|
| 1 | Ambiente e funções auxiliares |
| 2 | Carga da base, verificação dos requisitos mínimos e taxonomia |
| 3 | **Parte 1:** 7 consultas com `g.triples()` (T1 a T7) |
| 4 | Inferência com o reasoner OWL-RL e o efeito de cada construção OWL |
| 5 | **Parte 2:** 13 consultas SPARQL (S1 a S13): SELECT, ASK, CONSTRUCT e Update |
| 6 | Resumo das consultas |

Para executar, use **Executar tudo** (Colab) ou **Run All** (Jupyter). As células podem ser
reexecutadas em qualquer ordem: as atualizações da seção 5.4 trabalham sobre uma cópia do grafo.

## 1. Ambiente

A primeira célula instala `rdflib` e `owlrl` quando eles não estão disponíveis (caso do Colab). A
segunda baixa o `cinema.ttl` do GitHub quando o notebook roda fora do repositório clonado.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("rdflib") is None or importlib.util.find_spec("owlrl") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdflib", "owlrl"])

In [2]:
import urllib.request
from pathlib import Path

ARQUIVO_BASE = Path("cinema.ttl")
URL_BASE = "https://raw.githubusercontent.com/Nicolas-Kleiton/sbc-cinefilo-ontologia/main/cinema.ttl"

if not ARQUIVO_BASE.exists():  # no Colab, o .ttl ainda não está no disco
    urllib.request.urlretrieve(URL_BASE, ARQUIVO_BASE)
    print("cinema.ttl baixado do GitHub.")
else:
    print("Usando o cinema.ttl local.")

Usando o cinema.ttl local.


Funções auxiliares usadas no resto do notebook:

- `carregar_base()` lê o `.ttl` e devolve um grafo novo;
- `raciocinar(grafo)` aplica o fechamento dedutivo OWL-RL (`owlrl`) numa **cópia** do grafo;
- `nome(termo, grafo)` mostra o `rdfs:label` de um recurso em vez da URI;
- `mostrar(...)` imprime o resultado de um SELECT como tabela.

In [3]:
from decimal import Decimal

from owlrl import DeductiveClosure, OWLRL_Semantics
from rdflib import Graph, Literal, Namespace
from rdflib.namespace import OWL, RDF, RDFS, XSD

CINE = Namespace("http://ufpb.br/sbc/cinefilo#")
SCHEMA = Namespace("http://schema.org/")
# Vocabulário que o owlrl usa para registrar inconsistências encontradas
ERRO_OWLRL = Namespace("http://www.daml.org/2002/03/agents/agent-ont#")

# Prefixos passados ao rdflib em cada consulta SPARQL (initNs), para não repetir PREFIX.
PREFIXOS = {"cine": CINE, "schema": SCHEMA, "rdf": RDF, "rdfs": RDFS, "owl": OWL, "xsd": XSD}


def carregar_base():
    """Lê cinema.ttl e devolve um grafo com as triplas declaradas."""
    g = Graph()
    g.parse(ARQUIVO_BASE, format="turtle")
    g.bind("cine", CINE)
    g.bind("schema", SCHEMA, replace=True)  # sem replace, o rdflib usaria "schema1:"
    return g


def copiar(grafo):
    """Cópia independente de um grafo, com os mesmos prefixos."""
    novo = Graph()
    for prefixo, ns in grafo.namespaces():
        novo.bind(prefixo, ns, override=True, replace=True)
    novo += grafo
    return novo


def raciocinar(grafo):
    """Devolve uma cópia de `grafo` com todas as triplas deriváveis por OWL-RL.

    O owlrl também gera triplas que não interessam às consultas: literais como
    sujeito (ex.: "8.7" owl:sameAs "8.7") e a identidade reflexiva x owl:sameAs x.
    Elas são removidas para manter o grafo inferido legível.
    """
    inferido = copiar(grafo)
    DeductiveClosure(OWLRL_Semantics).expand(inferido)
    ruido = [t for t in inferido
             if isinstance(t[0], Literal) or (t[1] == OWL.sameAs and t[0] == t[2])]
    for t in ruido:
        inferido.remove(t)
    return inferido


def erros_de_consistencia(grafo_inferido):
    """Mensagens de inconsistência registradas pelo owlrl durante o raciocínio."""
    return sorted(str(m) for m in grafo_inferido.objects(None, ERRO_OWLRL.error))


def nome(termo, grafo):
    """rdfs:label do recurso; se não houver, o nome curto da URI."""
    if termo is None:
        return "—"
    if isinstance(termo, Literal):
        valor = termo.toPython()
        if isinstance(valor, (Decimal, float)):
            return f"{valor:.2f}"
        return str(valor)
    rotulo = grafo.value(termo, RDFS.label)
    return str(rotulo) if rotulo is not None else termo.n3(grafo.namespace_manager)


def nomes(termos, grafo):
    return sorted(nome(t, grafo) for t in termos)


def cabecalho(codigo, proposito):
    print(f"[Consulta {codigo}] {proposito}")
    print("-" * 78)


def mostrar(resultado, grafo):
    """Imprime o resultado de um SELECT como tabela de texto."""
    colunas = [str(v) for v in resultado.vars]
    linhas = [[nome(linha[c], grafo) for c in colunas] for linha in resultado]
    larguras = [max([len(c)] + [len(l[i]) for l in linhas]) for i, c in enumerate(colunas)]
    print("  ".join(c.ljust(w) for c, w in zip(colunas, larguras)))
    print("  ".join("-" * w for w in larguras))
    for linha in linhas:
        print("  ".join(v.ljust(w) for v, w in zip(linha, larguras)))
    print(f"({len(linhas)} linha(s))")

## 2. Carga da base e requisitos mínimos

A base é lida do arquivo Turtle, sem nenhuma tripla criada em código. A célula seguinte confere,
direto no grafo, cada requisito mínimo da especificação.

In [4]:
base = carregar_base()

classes = set(base.subjects(RDF.type, OWL.Class))
props_objeto = set(base.subjects(RDF.type, OWL.ObjectProperty))
props_dados = set(base.subjects(RDF.type, OWL.DatatypeProperty))
instancias = {s for s, o in base.subject_objects(RDF.type) if o in classes}

# Construções OWL: predicados do vocabulário owl: (inverseOf, disjointWith...) e
# características de propriedade (TransitiveProperty...). As declarações básicas
# (owl:Class, owl:ObjectProperty...) não contam como construção.
declaracoes = {OWL.Class, OWL.ObjectProperty, OWL.DatatypeProperty, OWL.Ontology}
construcoes_owl = {p for p in base.predicates() if p.startswith(OWL)}
construcoes_owl |= {o for o in base.objects(None, RDF.type) if o.startswith(OWL) and o not in declaracoes}

props_com_dominio_e_range = {p for p in props_objeto | props_dados
                             if (p, RDFS.domain, None) in base and (p, RDFS.range, None) in base}


def profundidade(classe):
    """Número de níveis entre a classe e a raiz da sua hierarquia."""
    pais = list(base.objects(classe, RDFS.subClassOf))
    return 0 if not pais else 1 + max(profundidade(p) for p in pais)


requisitos = [
    ("Classes", len(classes), 8),
    ("Níveis abaixo da raiz", max(profundidade(c) for c in classes), 2),
    ("Propriedades de objeto", len(props_objeto), 5),
    ("Propriedades de dados", len(props_dados), 5),
    ("Propriedades (total)", len(props_objeto | props_dados), 10),
    ("Instâncias", len(instancias), 25),
    ("Triplas", len(base), 50),
    ("Construções OWL", len(construcoes_owl), 5),
]
print(f"{'Requisito':<25}{'Na base':>9}{'Mínimo':>9}")
for rotulo, valor, minimo in requisitos:
    print(f"{rotulo:<25}{valor:>9}{minimo:>9}  {'ok' if valor >= minimo else 'FALTA'}")

print(f"\nPropriedades com domain e range: {len(props_com_dominio_e_range)} de {len(props_objeto | props_dados)}")
print("Construções OWL usadas:", ", ".join(sorted(c.n3(base.namespace_manager) for c in construcoes_owl)))

Requisito                  Na base   Mínimo
Classes                         18        8  ok
Níveis abaixo da raiz            3        2  ok
Propriedades de objeto          12        5  ok
Propriedades de dados            7        5  ok
Propriedades (total)            19       10  ok
Instâncias                      68       25  ok
Triplas                        561       50  ok
Construções OWL                  7        5  ok

Propriedades com domain e range: 19 de 19
Construções OWL usadas: owl:AsymmetricProperty, owl:FunctionalProperty, owl:SymmetricProperty, owl:TransitiveProperty, owl:disjointWith, owl:equivalentClass, owl:inverseOf


### Taxonomia

A árvore abaixo é gerada a partir das triplas `rdfs:subClassOf` do arquivo. São quatro raízes
independentes (Obra, Pessoa, Organizacao e Premio). A hierarquia mais profunda é a de filmes, com
três níveis abaixo de `Obra`.

In [5]:
def imprimir_arvore(classe, prefixo="", ultimo=True, raiz=True):
    rotulo = classe.n3(base.namespace_manager)
    n = sum(1 for _ in base.subjects(RDF.type, classe))
    sufixo = f"  ({n} instância(s) declarada(s))" if n else ""
    print((rotulo if raiz else prefixo + ("└─ " if ultimo else "├─ ") + rotulo) + sufixo)
    filhos = sorted(base.subjects(RDFS.subClassOf, classe))
    novo_prefixo = "" if raiz else prefixo + ("   " if ultimo else "│  ")
    for i, filho in enumerate(filhos):
        imprimir_arvore(filho, novo_prefixo, i == len(filhos) - 1, raiz=False)


raizes = sorted(c for c in classes if (c, RDFS.subClassOf, None) not in base)
for r in raizes:
    imprimir_arvore(r)

cine:Obra
└─ cine:Filme
   ├─ cine:Documentario  (1 instância(s) declarada(s))
   └─ cine:Ficcao
      ├─ cine:Acao  (2 instância(s) declarada(s))
      ├─ cine:Animacao  (3 instância(s) declarada(s))
      ├─ cine:Drama  (6 instância(s) declarada(s))
      ├─ cine:FiccaoCientifica  (7 instância(s) declarada(s))
      └─ cine:Terror  (2 instância(s) declarada(s))
cine:Organizacao
└─ cine:Estudio  (8 instância(s) declarada(s))
cine:Pessoa
└─ cine:Profissional
   ├─ cine:Ator  (19 instância(s) declarada(s))
   ├─ cine:Compositor  (4 instância(s) declarada(s))
   ├─ cine:Diretor  (10 instância(s) declarada(s))
   └─ cine:Roteirista  (3 instância(s) declarada(s))
cine:Premio  (7 instância(s) declarada(s))


## 3. Parte 1: consultas com `g.triples()`

`g.triples((s, p, o))` devolve as triplas que casam com o padrão, e `None` funciona como coringa.
As consultas desta seção rodam sobre o grafo **sem inferência** (`base`), ou seja, só sobre o que
está escrito no `.ttl`. Cada uma usa um padrão diferente:

| Consulta | Padrão | Pergunta |
|---|---|---|
| T1 | `(s, p, None)` | Quem está no elenco de Interestelar? |
| T2 | `(None, p, o)` | Que filmes Villeneuve dirigiu? |
| T3 | `(None, p, None)` | Quais são as maiores notas do IMDb? |
| T4 | `(s, None, None)` | O que a base diz sobre Clint Eastwood? |
| T5 | `(None, None, o)` | Onde Clint Eastwood aparece como objeto? |
| T6 | `(s, None, o)` | Que relação liga Nolan a Villeneuve? |
| T7 | `(None, rdf:type, o)` | Que filmes são do tipo `Ficcao`? |

In [6]:
# T1: padrão (sujeito fixo, predicado fixo, None)
# Busca:    os atores ligados a Interestelar por cine:temNoElenco.
# Esperado: Anne Hathaway, Jessica Chastain e Matthew McConaughey.
cabecalho("T1", "Elenco de Interestelar  (s, p, None)")
for ator in nomes((o for _, _, o in base.triples((CINE.interestelar, CINE.temNoElenco, None))), base):
    print(" ", ator)

[Consulta T1] Elenco de Interestelar  (s, p, None)
------------------------------------------------------------------------------
  Anne Hathaway
  Jessica Chastain
  Matthew McConaughey


In [7]:
# T2: padrão (None, predicado fixo, objeto fixo)
# Busca:    os filmes cujo cine:dirigidoPor é Denis Villeneuve.
# Esperado: A Chegada, Blade Runner 2049, Duna e Duna: Parte Dois.
cabecalho("T2", "Filmes dirigidos por Denis Villeneuve  (None, p, o)")
for titulo in nomes((s for s, _, _ in base.triples((None, CINE.dirigidoPor, CINE.villeneuve))), base):
    print(" ", titulo)

[Consulta T2] Filmes dirigidos por Denis Villeneuve  (None, p, o)
------------------------------------------------------------------------------
  A Chegada
  Blade Runner 2049
  Duna
  Duna: Parte Dois


In [8]:
# T3: padrão (None, predicado fixo, None)
# Busca:    todas as triplas cine:notaIMDb, ou seja, a nota de cada filme. Depois
#           ordena em Python e mostra as cinco maiores.
# Esperado: 17 notas; no topo, Batman: O Cavaleiro das Trevas (9.0), A Origem (8.8),
#           Interestelar (8.7), Duna: Parte Dois (8.5) e Parasita (8.5).
cabecalho("T3", "Cinco maiores notas no IMDb  (None, p, None)")
notas = [(o.toPython(), nome(s, base)) for s, _, o in base.triples((None, CINE.notaIMDb, None))]
print(f"  {len(notas)} filmes com nota registrada")
for nota, titulo in sorted(notas, key=lambda x: (-x[0], x[1]))[:5]:
    print(f"  {nota:>4}  {titulo}")

[Consulta T3] Cinco maiores notas no IMDb  (None, p, None)
------------------------------------------------------------------------------
  17 filmes com nota registrada
   9.0  Batman: O Cavaleiro das Trevas
   8.8  A Origem
   8.7  Interestelar
   8.5  Duna: Parte Dois
   8.5  Parasita


In [9]:
# T4: padrão (sujeito fixo, None, None)
# Busca:    todas as triplas em que Clint Eastwood é o sujeito (o "perfil" dele).
# Esperado: tipo Diretor, nome, ano de nascimento 1930, nacionalidade EUA e a
#           colaboração com Morgan Freeman. Nada diz que ele é ator: esse fato está
#           em triplas onde ele aparece como objeto (consulta T5).
cabecalho("T4", "Tudo o que é afirmado sobre Clint Eastwood  (s, None, None)")
for s, p, o in sorted(base.triples((CINE.eastwood, None, None))):
    print(f"  {p.n3(base.namespace_manager):<22} {nome(o, base)}")

[Consulta T4] Tudo o que é afirmado sobre Clint Eastwood  (s, None, None)
------------------------------------------------------------------------------
  cine:anoNascimento     1930
  cine:colaborouCom      Morgan Freeman
  cine:nacionalidade     EUA
  rdf:type               Diretor
  rdfs:label             Clint Eastwood


In [10]:
# T5: padrão (None, None, objeto fixo)
# Busca:    todas as triplas em que Clint Eastwood é o objeto.
# Esperado: Menina de Ouro aponta para ele duas vezes, por cine:dirigidoPor e por
#           cine:temNoElenco: ele dirigiu e atuou no mesmo filme.
cabecalho("T5", "Onde Clint Eastwood aparece como objeto  (None, None, o)")
for s, p, o in sorted(base.triples((None, None, CINE.eastwood))):
    print(f"  {nome(s, base):<16} {p.n3(base.namespace_manager)}")

[Consulta T5] Onde Clint Eastwood aparece como objeto  (None, None, o)
------------------------------------------------------------------------------
  Menina de Ouro   cine:dirigidoPor
  Menina de Ouro   cine:temNoElenco


In [11]:
# T6: padrão (sujeito fixo, None, objeto fixo)
# Busca:    qualquer predicado que ligue Nolan diretamente a Villeneuve.
# Esperado: uma única tripla, cine:influenciou.
cabecalho("T6", "Que relação liga Christopher Nolan a Denis Villeneuve  (s, None, o)")
for s, p, o in base.triples((CINE.nolan, None, CINE.villeneuve)):
    print(" ", p.n3(base.namespace_manager))

[Consulta T6] Que relação liga Christopher Nolan a Denis Villeneuve  (s, None, o)
------------------------------------------------------------------------------
  cine:influenciou


In [12]:
# T7: padrão (None, rdf:type, objeto fixo)
# Busca:    as instâncias declaradas como cine:Ficcao.
# Esperado: NENHUMA. Os filmes declaram só o gênero mais específico (Drama, Terror...).
#           Que um Drama também é Ficcao depende de rdfs:subClassOf, e g.triples() só
#           enxerga o que está escrito. Na seção 4, o reasoner resolve isso.
cabecalho("T7", "Filmes do tipo Ficcao, sem inferência  (None, rdf:type, o)")
ficcao = [s for s, _, _ in base.triples((None, RDF.type, CINE.Ficcao))]
print(f"  {len(ficcao)} resultado(s)")

[Consulta T7] Filmes do tipo Ficcao, sem inferência  (None, rdf:type, o)
------------------------------------------------------------------------------
  0 resultado(s)
